# Notebook 5 - Portfolio Optimization & Monte Carlo Simulation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

returns = pd.read_csv("data/processed/stock_returns.csv", index_col=0, parse_dates=True)

annual_returns = returns.mean()*252
cov_matrix = returns.cov()*252

n_assets = len(returns.columns)


## Maximum Sharpe Portfolio

In [ ]:
def neg_sharpe(weights):
    r = np.dot(weights, annual_returns)
    v = np.sqrt(weights.T @ cov_matrix @ weights)
    return -(r-0.06)/v

constraints = ({'type':'eq','fun':lambda w: np.sum(w)-1})
bounds = tuple((0,1) for _ in range(n_assets))
x0 = np.repeat(1/n_assets,n_assets)

opt = minimize(neg_sharpe, x0, method="SLSQP",
               bounds=bounds,
               constraints=constraints)

max_sharpe_weights = pd.Series(opt.x,index=returns.columns)
max_sharpe_weights


## Minimum Variance Portfolio

In [ ]:
def variance(weights):
    return weights.T @ cov_matrix @ weights

opt2 = minimize(variance, x0,
                method="SLSQP",
                bounds=bounds,
                constraints=constraints)

min_var_weights = pd.Series(opt2.x,index=returns.columns)
min_var_weights


## Efficient Frontier

In [ ]:
sim=[]
for _ in range(5000):
    w=np.random.random(n_assets)
    w=w/w.sum()
    ret=np.dot(w,annual_returns)
    vol=np.sqrt(w.T@cov_matrix@w)
    sharpe=(ret-0.06)/vol
    sim.append([ret,vol,sharpe])

frontier=pd.DataFrame(sim,columns=["Return","Volatility","Sharpe"])

plt.figure(figsize=(8,6))
plt.scatter(frontier["Volatility"],frontier["Return"],
            c=frontier["Sharpe"],s=8)
plt.xlabel("Volatility")
plt.ylabel("Return")
plt.title("Efficient Frontier")
plt.colorbar(label="Sharpe")
plt.show()


## Monte Carlo Simulation

In [ ]:
portfolio_returns = returns.dot(max_sharpe_weights)

mu = portfolio_returns.mean()
sigma = portfolio_returns.std()

days=252
paths=10000
initial=1000000

sim_paths=np.zeros((days,paths))
sim_paths[0]=initial

for p in range(paths):
    for d in range(1,days):
        sim_paths[d,p]=sim_paths[d-1,p]*(1+np.random.normal(mu,sigma))

plt.figure(figsize=(10,5))
plt.plot(sim_paths[:,:100],alpha=.25)
plt.title("Monte Carlo Simulation (100 of 10,000 paths)")
plt.xlabel("Trading Days")
plt.ylabel("Portfolio Value")
plt.show()


In [ ]:
ending=sim_paths[-1]

summary=pd.DataFrame({
    "Metric":[
        "Median Ending Value",
        "Mean Ending Value",
        "5th Percentile",
        "95th Percentile",
        "Probability of Profit"
    ],
    "Value":[
        np.median(ending),
        ending.mean(),
        np.percentile(ending,5),
        np.percentile(ending,95),
        (ending>initial).mean()
    ]
})

summary


In [ ]:
max_sharpe_weights.to_csv("output/max_sharpe_weights.csv")
min_var_weights.to_csv("output/min_variance_weights.csv")
frontier.to_csv("output/efficient_frontier.csv",index=False)
summary.to_csv("output/monte_carlo_summary.csv",index=False)

pd.DataFrame(sim_paths).to_csv("output/monte_carlo_paths.csv",index=False)


## Next Notebook

Notebook 6 will create:

- AI Portfolio Risk Score
- Risk attribution
- Executive summary
- Power BI export tables
- Dashboard-ready datasets
- Final report
